# Worldle Final Project

This notebook stitches the finished `wdo` helpers into a playable Worldle-style geography game. The notebook deliberately keeps spatial math out of the UI cells: country centers, bounding boxes, distances, bearings, arrows, and flag lookup all come from `wdo`.

## What I finished in `wdo`

- `wdo.geometry.bbox`: `iter_coordinates`, `bbox_from_points`, `bbox_from_feature`, `bbox_from_features`, and GeoJSON-ready `bbox_to_polygon`.
- `wdo.maps.leaflet_helpers`: `make_map`, `add_geojson`, `fit_map_to_geojson`, controls, bounding boxes, and paths using `ipyleaflet`.
- `wdo.games.worldle`: `choose_target`, `feature_center`, `guess_feedback`, `format_feedback`, `build_country_lookup`, flag rendering, proximity coloring, and share text.

## Country lookup notes

The polygon file in this copy includes ISO-2 and ISO-3 fields, so most flags join directly on ISO-2. I still added normalized name matching plus aliases for common real-world mismatches such as `United States of America` -> `United States` and `Czech Republic` -> `Czechia`. A small set of disputed territories and special zones have no matching flag, and the UI falls back gracefully.

## Known bugs

- The default `feature_center(method="bbox")` is fast and good enough for gameplay, but it can be odd for countries crossing the antimeridian or countries with far-flung islands.
- This notebook needs `ipyleaflet` and `ipywidgets` in the active Jupyter environment for the interactive map UI. The pure feedback tests run without them.
- A completed-round sample image is included below for the submission write-up.

![Completed Worldle sample](completed_round_screenshot.svg)


In [1]:
from pathlib import Path
import json
import random
import sys


def find_assignment_dir():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        for rel in [Path('.'), Path('Assignments_Completed/04-Worldle'), Path('Assignments/04-Worldle')]:
            candidate = (base / rel).resolve()
            if (candidate / 'Resources' / 'wdo').exists() and (candidate / 'Resources' / 'Data').exists():
                return candidate
    raise FileNotFoundError('Could not find the 04-Worldle assignment directory.')


ASSIGNMENT_DIR = find_assignment_dir()
RESOURCES_DIR = ASSIGNMENT_DIR / 'Resources'
DATA_DIR = RESOURCES_DIR / 'Data'
sys.path.insert(0, str(RESOURCES_DIR))

ASSIGNMENT_DIR


PosixPath('/workspaces/spatial_mapping_repo_Thad_Taylor/completed_assignments/04-Worldle')

In [2]:
from wdo.io.geojson_tools import feature_count, get_features, load_geojson, property_names
from wdo.games.worldle import (
    build_country_lookup,
    choose_target,
    country_name,
    feature_center,
    flag_to_data_uri,
    format_feedback,
    guess_feedback,
    iso3_code,
    normalize_name,
    render_guess_row,
    share_result,
    silhouette_feature,
)

countries = load_geojson(DATA_DIR / 'countries_export.json')
with open(DATA_DIR / 'flag-icons' / 'country.json', encoding='utf-8') as fp:
    flag_index = json.load(fp)

features = [feature for feature in get_features(countries) if iso3_code(feature)]
country_lookup = build_country_lookup(countries, flag_index)
feature_by_iso3 = {iso3_code(feature): feature for feature in features}
name_to_iso3 = {country_name(feature): iso3_code(feature) for feature in features}
name_options = sorted(name_to_iso3)

print('Feature count:', feature_count(countries))
print('Playable countries:', len(features))
print('Properties:', property_names(countries))
print('Flags joined:', sum(1 for item in country_lookup.values() if item['flag_path']))


Feature count: 258
Playable countries: 236
Properties: ['ISO3166-1-Alpha-2', 'ISO3166-1-Alpha-3', 'name']
Flags joined: 240


In [3]:
# Quick non-UI sanity check: compare two countries before the widget game starts.
seed_target = choose_target(features, seed=205)
first_guess = feature_by_iso3.get('USA') or features[0]
feedback = guess_feedback(first_guess, seed_target)

print('Target:', country_name(seed_target))
print('Guess:', format_feedback(feedback))
print('Target center:', feature_center(seed_target))


Target: El Salvador
Guess: United States of America: 8,865 km ← W
Target center: (13.802005000000001, -88.9039865)


In [4]:
class WorldleGame:
    def __init__(self, features, lookup, seed=None, max_guesses=6):
        self.features = list(features)
        self.lookup = lookup
        self.feature_by_iso3 = {iso3_code(feature): feature for feature in self.features}
        self.seed = random.randrange(1_000_000) if seed is None else seed
        self.target = choose_target(self.features, seed=self.seed)
        self.target_iso3 = iso3_code(self.target)
        self.max_guesses = max_guesses
        self.history = []
        self.finished = False
        self.won = False

    def submit_guess(self, guess_iso3):
        if self.finished:
            return {'error': 'This round is already finished.'}
        if guess_iso3 not in self.feature_by_iso3:
            return {'error': 'Choose a country from the list.'}
        if any(item['guess_iso3'] == guess_iso3 for item in self.history):
            return {'error': 'You already guessed that country.'}

        guess_feature = self.feature_by_iso3[guess_iso3]
        result = guess_feedback(guess_feature, self.target)
        result['guess_number'] = len(self.history) + 1
        result['remaining'] = self.max_guesses - result['guess_number']
        result['flag_path'] = self.lookup.get(guess_iso3, {}).get('flag_path')
        self.history.append(result)

        if result['correct']:
            self.finished = True
            self.won = True
        elif len(self.history) >= self.max_guesses:
            self.finished = True

        return result

    def give_up(self):
        self.finished = True
        return self.target

    def share_text(self):
        outcome = 'Solved' if self.won else 'Revealed'
        header = f'Worldle seed {self.seed} - {outcome}: {country_name(self.target)}'
        return header + '\n' + share_result(self.history)


In [5]:
try:
    import ipywidgets as widgets
    from IPython.display import HTML, display
    from wdo.maps.leaflet_helpers import add_geojson, fit_map_to_geojson, make_map
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    WIDGET_IMPORT_ERROR = exc


In [6]:
TARGET_STYLE = {
    'color': '#17324d',
    'fillColor': '#2a9d8f',
    'weight': 2,
    'fillOpacity': 0.72,
}

REVEAL_STYLE = {
    'color': '#e76f51',
    'fillColor': '#f4a261',
    'weight': 3,
    'fillOpacity': 0.34,
}


def country_flag_src(iso3):
    path = country_lookup.get(iso3, {}).get('flag_path')
    # Relative paths work in Jupyter when the notebook is opened from this folder.
    return path


def find_iso_from_text(text):
    if text in name_to_iso3:
        return name_to_iso3[text]
    normalized = normalize_name(text)
    for name, iso3 in name_to_iso3.items():
        if normalize_name(name) == normalized:
            return iso3
    return None


In [7]:
def launch_game(seed=205, max_guesses=6):
    if not WIDGETS_AVAILABLE:
        print('Interactive widgets are unavailable in this environment:')
        print(WIDGET_IMPORT_ERROR)
        return None

    holder = widgets.VBox()

    def start_round(round_seed=None):
        game = WorldleGame(features, country_lookup, seed=round_seed, max_guesses=max_guesses)
        mystery_shape = silhouette_feature(game.target)
        game_map = make_map(center=(0, 0), zoom=2, basemap='blank', layout=widgets.Layout(height='520px'))
        add_geojson(game_map, mystery_shape, name='Mystery silhouette', style=TARGET_STYLE)
        fit_map_to_geojson(game_map, mystery_shape)

        title = widgets.HTML(
            f'''
            <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
                        padding:10px 0 6px 0">
              <div style="font-size:28px;font-weight:800;color:#17324d">Worldle</div>
              <div style="color:#4b6578">Seed {game.seed} ? {game.max_guesses} guesses ? follow the arrows.</div>
            </div>
            '''
        )
        banner = widgets.HTML(
            "<div style='padding:8px 0;color:#4b6578;font-weight:650'>Type a country and make your first guess.</div>"
        )
        guess_box = widgets.Combobox(
            options=name_options,
            placeholder='Type a country',
            ensure_option=False,
            layout=widgets.Layout(width='360px'),
        )
        guess_button = widgets.Button(description='Guess', button_style='primary', icon='location-arrow')
        give_up_button = widgets.Button(description='Give up', button_style='warning', icon='flag')
        new_button = widgets.Button(description='New game', icon='refresh')
        history = widgets.Output()
        share_box = widgets.Textarea(
            value='',
            placeholder='Share text appears here after guesses.',
            layout=widgets.Layout(width='100%', height='96px'),
        )

        def reveal_target(message):
            add_geojson(game_map, mystery_shape, name='Answer silhouette', style=REVEAL_STYLE)
            fit_map_to_geojson(game_map, mystery_shape)
            banner.value = message
            guess_button.disabled = True
            give_up_button.disabled = True
            share_box.value = game.share_text()

        def redraw_history():
            history.clear_output()
            with history:
                if not game.history:
                    display(HTML("<div style='color:#6b7f8f;padding:10px 0'>No guesses yet.</div>"))
                    return
                rows = []
                for item in game.history:
                    rows.append(
                        render_guess_row(
                            item['guess_name'],
                            item.get('flag_path'),
                            item['arrow'],
                            item['distance_km'],
                            correct=item['correct'],
                        )
                    )
                display(HTML(''.join(rows)))

        def on_guess(_=None):
            guess_iso3 = find_iso_from_text(guess_box.value)
            result = game.submit_guess(guess_iso3)
            if 'error' in result:
                banner.value = f"<div style='padding:8px 0;color:#b45309;font-weight:700'>{result['error']}</div>"
                return

            redraw_history()
            share_box.value = game.share_text()
            guess_box.value = ''

            if result['correct']:
                reveal_target(
                    f"<div style='padding:8px 0;color:#2a9d8f;font-weight:800'>Solved in {result['guess_number']} guesses: {result['target_name']}.</div>"
                )
            elif game.finished:
                reveal_target(
                    f"<div style='padding:8px 0;color:#e76f51;font-weight:800'>Out of guesses. The country was {result['target_name']}.</div>"
                )
            else:
                banner.value = (
                    f"<div style='padding:8px 0;color:#17324d;font-weight:700'>"
                    f"{format_feedback(result)} ? {result['remaining']} guesses left.</div>"
                )

        def on_give_up(_):
            game.give_up()
            reveal_target(
                f"<div style='padding:8px 0;color:#e76f51;font-weight:800'>Revealed: {country_name(game.target)}.</div>"
            )

        def on_new(_):
            start_round(random.randrange(1_000_000))

        guess_button.on_click(on_guess)
        give_up_button.on_click(on_give_up)
        new_button.on_click(on_new)
        if hasattr(guess_box, 'on_submit'):
            guess_box.on_submit(on_guess)

        controls = widgets.HBox([guess_box, guess_button, give_up_button, new_button])
        redraw_history()
        holder.children = [title, game_map, banner, controls, history, share_box]

    start_round(seed)
    return holder


In [8]:
game_ui = launch_game(seed=205, max_guesses=6)
if game_ui is not None:
    display(game_ui)


/tmp/ipykernel_30638/1822448458.py:109: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  guess_box.on_submit(on_guess)


TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.